# Benchmark Dataset Creation (Kaggle Notebook)
- Use `GPU T4 x 2` Acceleration
- Use `Internet access`

In [ ]:
from datasets import load_dataset

LANG = "en" # "ja"
ds = load_dataset("google/wiki40b", LANG)
ds

In [ ]:
import re

import pandas as pd
import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def parse_wiki40b_article(text: str):
    title = ""
    section = ""
    paragraphs = []
    mode = None
    for part in re.split(
        r"(_START_ARTICLE_|_START_SECTION_|_START_PARAGRAPH_)", text
    ):
        part = part.strip()
        if not part:
            continue
        if part == "_START_ARTICLE_":
            mode = "title"
            continue
        if part == "_START_SECTION_":
            mode = "section"
            continue
        if part == "_START_PARAGRAPH_":
            mode = "paragraph"
            continue

        content = part.replace("_NEWLINE_", "\n").replace("\xa0", " ")
        content = re.sub(r"\n+", "\n", content).strip()

        if mode == "title":
            title = content
        elif mode == "section":
            section = content
        elif mode == "paragraph":
            paragraphs.append((section, content))
        mode = None
    return title, paragraphs
    

def build_chunk_text(title, section, paragraph):
    parts = []

    if title:
        parts.append(title)

    if section:
        parts.append(section)

    if paragraph:
        parts.append(paragraph)

    return "\n\n".join(parts)


def chunk_text_by_chars(text, max_chars=2000):
    return [
        text[i:i+max_chars]
        for i in range(0, len(text), max_chars)
    ]

def chunk_wiki40b_article(text, max_chars=2000):
    title, paragraphs = parse_wiki40b_article(text)

    chunks = []

    for section, paragraph in paragraphs:
        for body in chunk_text_by_chars(
            paragraph,
            max_chars,
        ):
            chunk = build_chunk_text(title,section,body)
            tokens = tokenizer.encode(chunk, add_special_tokens=True)
            if len(tokens) > 800:
                print(f"Too big chunk found. Skipped: ", chunk)
                continue
            chunks.append(chunk)
            
    return chunks


In [ ]:
from sentence_transformers import SentenceTransformer
if "model" not in globals():
    print("Loading model...")
    model = SentenceTransformer(MODEL_NAME)
    
pool = model.start_multi_process_pool(
    target_devices=["cuda:0", "cuda:1"]
)

N = 1_000_000

BUFFER_SIZE = 2048 * 2
batch_size = 64

# shuffle and choose N rows
print("Shuffling dataset...")
sample_ds = ds["train"].shuffle(seed=42).select(range(N))

rows = []
buffer_chunks = []
total_embeddings = 0
pbar = tqdm(total=N, desc="Embedding chunks")

for article in sample_ds["text"]:
    buffer_chunks.extend(chunk_wiki40b_article(article))

    if len(buffer_chunks) >= BUFFER_SIZE:
        embeddings = model.encode(
            buffer_chunks,
            pool=pool,
            chunk_size=256,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        
        n = len(buffer_chunks)
        pbar.update(n)

        for chunk, emb in zip(buffer_chunks, embeddings):
            rows.append({"text": chunk, "embedding": emb.tolist()})

        buffer_chunks.clear()
        if len(rows) >= N:
            break
            
model.stop_multi_process_pool(pool)

df = pd.DataFrame(rows)

df.to_parquet(
    f"/kaggle/working/wiki40b_en_embeddings_{N}.parquet",
    index=False,
)